# P9 FINAL: clean retrain (3 det + 1 functional MC) + full eval
Self-consistent locked-env run replacing all published numbers. Data/metrics cells verbatim from original notebook.

In [ ]:

import os, subprocess
_q = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True)
_sm = _q.stdout.strip().split(".")
SM = (int(_sm[0]), int(_sm[1])) if len(_sm) == 2 and _sm[0].strip().isdigit() else (9, 0)
print("GPU SM:", SM)
if SM < (7, 0):
    subprocess.run(["pip", "install", "-q", "torch==2.3.1+cu118", "torchvision==0.18.1+cu118",
                    "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
os.system("pip install -q pydicom albumentations pretrainedmodels efficientnet_pytorch tqdm munch scikit-learn")
os.system("pip install -q --no-deps segmentation-models-pytorch")
import sys, time, math, glob, ast, shutil, json, random
import numpy as np, pandas as pd, pydicom, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Tuple, Optional, Any
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
print("torch:", torch.__version__, "| smp:", smp.__version__, "| alb:", A.__version__)
device = torch.device("cuda")
seed_everything = lambda s=42: (random.seed(s), np.random.seed(s), torch.manual_seed(s),
                                torch.cuda.manual_seed_all(s))


In [ ]:

df_splits = pd.read_csv(glob.glob("/kaggle/input/**/patient_splits.csv", recursive=True)[0])
dmap0 = {}
for p in glob.glob("/kaggle/input/**/dicom-images-train/**/*.dcm", recursive=True):
    dmap0.setdefault(os.path.basename(p).replace(".dcm", ""), p)
df_splits["dcm_path"] = df_splits["ImageId"].map(dmap0)
print(len(df_splits), "missing:", int(df_splits["dcm_path"].isna().sum()))


In [ ]:
# 3. SIIM-ACR RLE Decoder with Multi-Mask Logical OR Aggregation

def rle_decode(rle_str: Any, shape: tuple = (1024, 1024)) -> np.ndarray:
    """Decode SIIM-ACR Run-Length Encoded string into binary mask (Fortran order)."""
    if rle_str is None or (isinstance(rle_str, float) and np.isnan(rle_str)):
        return np.zeros(shape, dtype=np.uint8)

    rle_str = str(rle_str).strip()
    if rle_str == "" or rle_str == "-1":
        return np.zeros(shape, dtype=np.uint8)

    s = rle_str.split()
    starts = np.asarray([int(float(x)) for x in s[0::2]], dtype=int) - 1
    lengths = np.asarray([int(float(x)) for x in s[1::2]], dtype=int)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order="F")

def aggregate_rle_list(rle_list: List[str], shape: tuple = (1024, 1024)) -> np.ndarray:
    """Combine multiple RLE instances for an image using logical OR."""
    composite = np.zeros(shape, dtype=np.uint8)
    for rle in rle_list:
        mask = rle_decode(rle, shape=shape)
        composite = np.bitwise_or(composite, mask)
    return composite

# Verify RLE decoder on sample
sample_rles = ast.literal_eval(df_splits.iloc[0]["EncodedPixelsList"])
sample_mask = aggregate_rle_list(sample_rles)
print(f"Verification: sample mask shape={sample_mask.shape}, sum={np.sum(sample_mask)}")


In [ ]:
# 4. PyTorch Dataset and Radiologically Sound Augmentations

class SIIMPneumothoraxDataset(Dataset):
    """Dataset for SIIM-ACR pneumothorax radiographs with dense masks."""

    def __init__(self, df: pd.DataFrame, image_size: int = 512, transforms: Optional[Any] = None):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        dcm_path = row["dcm_path"]
        
        # Read DICOM
        try:
            dcm = pydicom.dcmread(dcm_path)
            img = dcm.pixel_array.astype(np.float32)
            if hasattr(dcm, "PhotometricInterpretation") and dcm.PhotometricInterpretation == "MONOCHROME1":
                img = np.amax(img) - img
        except Exception:
            img = np.zeros((1024, 1024), dtype=np.float32)

        # Normalize to uint8 [0, 255]
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = ((img - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
        else:
            img = np.zeros_like(img, dtype=np.uint8)

        # Grayscale to 3-channel
        img_3c = np.repeat(np.expand_dims(img, axis=-1), 3, axis=-1)

        # Decode ground-truth mask
        rle_raw = row["EncodedPixelsList"]
        if isinstance(rle_raw, str):
            try:
                rle_list = ast.literal_eval(rle_raw)
            except Exception:
                rle_list = [rle_raw]
        else:
            rle_list = [str(rle_raw)]

        mask = aggregate_rle_list(rle_list, shape=img.shape)

        # Apply Albumentations transforms
        if self.transforms is not None:
            augmented = self.transforms(image=img_3c, mask=mask)
            image_tensor = augmented["image"]
            mask_tensor = augmented["mask"].unsqueeze(0).float()
        else:
            image_tensor = torch.from_numpy(img_3c).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "image_id": row["ImageId"],
            "patient_id": str(row["PatientID"]),
            "view_position": str(row["ViewPosition"]),
            "has_pneumothorax": int(row["HasPneumothorax"])
        }

# Data augmentations: HorizontalFlip, ShiftScaleRotate, RandomBrightnessContrast
# Strictly NO VerticalFlip (violates anatomical cephalocaudal orientation)
train_transforms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-10, 10), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Partition DataFrames
# Train: Folds 1, 2, 3, 4 | Val: Fold 0 | Test Holdout: Fold 'test'
train_df = df_splits[df_splits["Fold"].isin(["1", "2", "3", "4", 1, 2, 3, 4])].reset_index(drop=True)
val_df = df_splits[df_splits["Fold"].isin(["0", 0])].reset_index(drop=True)
test_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)

print(f"Train set: {len(train_df)} images ({train_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Validation set: {len(val_df)} images ({val_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Test Holdout set: {len(test_df)} images ({test_df['HasPneumothorax'].mean()*100:.1f}% positive)")

# Verify zero patient leakage
train_val_patients = set(train_df["PatientID"]).union(set(val_df["PatientID"]))
test_patients = set(test_df["PatientID"])
overlap = train_val_patients.intersection(test_patients)
assert len(overlap) == 0, f"DATA LEAKAGE DETECTED: {len(overlap)} overlapping patients!"
print(f"LEAKAGE PROOF VERIFIED: Exactly {len(overlap)} overlapping patients between train/val and test.")


In [ ]:
# 5. Architecture: ResNet34 U-Net with Spatial Dropout and Combined Loss
import segmentation_models_pytorch as smp

class PneumothoraxUNet(nn.Module):
    """ResNet34 U-Net supporting deterministic inference and MC Dropout."""

    def __init__(self, dropout_rate: float = 0.2, pretrained: bool = True):
        super().__init__()
        self.dropout_rate = dropout_rate
        weights = "imagenet" if pretrained else None
        
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            decoder_channels=(256, 128, 64, 32, 16)
        )
        
        # Inject SpatialDropout2d into decoder blocks for MC Dropout
        if dropout_rate > 0.0:
            for idx in range(len(self.model.decoder.blocks)):
                self.model.decoder.blocks[idx].add_module(
                    "spatial_dropout", nn.Dropout2d(p=dropout_rate)
                )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    @torch.no_grad()
    def predict_deterministic(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Deterministic forward pass (eval mode)."""
        self.eval()
        logits = self.forward(x)
        prob = torch.sigmoid(logits)
        eps = 1e-7
        p_clamped = torch.clamp(prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        return {"prob": prob, "entropy": entropy}

    @torch.no_grad()
    def predict_mc_dropout(self, x: torch.Tensor, num_samples: int = 20) -> Dict[str, torch.Tensor]:
        """Monte Carlo Dropout inference with T stochastic passes."""
        self.train() # Activates Spatial Dropout during inference
        samples = []
        for _ in range(num_samples):
            logits = self.forward(x)
            samples.append(torch.sigmoid(logits))
        
        stacked = torch.stack(samples, dim=0) # (T, B, 1, H, W)
        mean_prob = torch.mean(stacked, dim=0)
        variance = torch.var(stacked, dim=0, unbiased=True)
        
        eps = 1e-7
        p_clamped = torch.clamp(mean_prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        
        return {
            "mean": mean_prob,
            "variance": variance,
            "entropy": entropy
        }

# Combined Loss: 0.5 * BCE + 0.5 * SoftDice
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        cardinality = (probs.pow(2) + targets.pow(2)).sum()
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice

class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice_loss = SoftDiceLoss(smooth=1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_bce = F.binary_cross_entropy_with_logits(logits, targets)
        loss_dice = self.dice_loss(logits, targets)
        return self.bce_weight * loss_bce + self.dice_weight * loss_dice

criterion = CombinedBCEDiceLoss()
print("Model architecture and CombinedBCEDiceLoss verified.")


In [ ]:
# 7. Evaluation Metrics: Disaggregated Dice, ESCE, AUROC-ED, and AURC

def compute_dice_coefficient(y_true: np.ndarray, y_pred: np.ndarray, empty_score: float = 1.0) -> float:
    y_true_sum = np.sum(y_true)
    y_pred_sum = np.sum(y_pred)
    if y_true_sum == 0 and y_pred_sum == 0:
        return empty_score
    if y_true_sum == 0 or y_pred_sum == 0:
        return 0.0
    intersection = np.sum(y_true * y_pred)
    return float(2.0 * intersection / (y_true_sum + y_pred_sum))

def compute_segmentation_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_pred_prob >= threshold).astype(np.uint8)
    y_true = (y_true > 0).astype(np.uint8)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    is_positive = float(np.sum(y_true) > 0)
    dice = compute_dice_coefficient(y_true, y_pred)
    iou = float(tp / (tp + fp + fn)) if (tp + fp + fn) > 0 else (1.0 if is_positive == 0 else 0.0)
    sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else (1.0 if np.sum(y_pred) == 0 else 0.0)

    return {
        "dice": dice,
        "iou": iou,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "is_positive": is_positive,
    }

def compute_esce(probs: np.ndarray, targets: np.ndarray, n_bins: int = 10):
    probs_flat = probs.flatten()
    targets_flat = targets.flatten()
    total = len(probs_flat)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_confs = np.zeros(n_bins)
    bin_accs = np.zeros(n_bins)
    bin_counts = np.zeros(n_bins)
    esce = 0.0

    for i in range(n_bins):
        low, high = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (probs_flat > low) & (probs_flat <= high) if i > 0 else (probs_flat >= low) & (probs_flat <= high)
        count = np.sum(mask)
        bin_counts[i] = count
        if count > 0:
            conf = np.mean(probs_flat[mask])
            acc = np.mean(targets_flat[mask])
            bin_confs[i] = conf
            bin_accs[i] = acc
            esce += (count / total) * np.abs(acc - conf)

    return float(esce), bin_confs, bin_accs, bin_counts

def compute_brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs - targets) ** 2))

def compute_error_detection_auroc(uncertainty_map: np.ndarray, probs: np.ndarray, targets: np.ndarray, threshold: float = 0.5) -> float:
    preds = (probs >= threshold).astype(np.uint8)
    error = np.abs(targets - preds).flatten()
    uncertainty = uncertainty_map.flatten()
    if np.all(error == 0) or np.all(error == 1):
        return 0.5
    if len(error) > 100_000:
        idx = np.random.choice(len(error), size=100_000, replace=False)
        error = error[idx]
        uncertainty = uncertainty[idx]
    try:
        return float(roc_auc_score(error, uncertainty))
    except Exception:
        return 0.5

def aggregate_case_uncertainty(uncertainty_map: np.ndarray, k: int = 500) -> float:
    flat = uncertainty_map.flatten()
    k = min(k, len(flat))
    top_k_vals = np.partition(flat, -k)[-k:]
    return float(np.mean(top_k_vals))

def compute_risk_coverage_curve(case_uncertainties: np.ndarray, case_risks: np.ndarray, steps: int = 50, min_cov: float = 0.20):
    n = len(case_uncertainties)
    sorted_idx = np.argsort(case_uncertainties)
    coverages = np.linspace(min_cov, 1.0, steps)
    risks = np.zeros(steps)
    for i, cov in enumerate(coverages):
        retain_n = max(1, int(round(cov * n)))
        retained = sorted_idx[:retain_n]
        risks[i] = float(np.mean(case_risks[retained]))
    return coverages, risks

def compute_aurc(coverages: np.ndarray, risks: np.ndarray) -> float:
    cov_norm = (coverages - coverages[0]) / (coverages[-1] - coverages[0])
    try:
        from scipy.integrate import trapezoid
        return float(trapezoid(risks, cov_norm))
    except ImportError:
        trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
        return float(trapz_fn(risks, cov_norm))


In [ ]:

# ---- splits / loaders (verbatim transforms) ----
F_ = lambda v: df_splits["Fold"].isin([v, int(v)])
tr_df = df_splits[F_("1") | F_("2") | F_("3") | F_("4")].reset_index(drop=True)
va_df = df_splits[F_("0")].reset_index(drop=True)
te_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)
print(len(tr_df), len(va_df), len(te_df))
assert (set(tr_df["PatientID"]) | set(va_df["PatientID"])).isdisjoint(set(te_df["PatientID"]))
tr_ld = DataLoader(SIIMPneumothoraxDataset(tr_df, image_size=512, transforms=train_transforms),
                   batch_size=16, shuffle=True, num_workers=2)
va_ld = DataLoader(SIIMPneumothoraxDataset(va_df, image_size=512, transforms=val_transforms),
                   batch_size=16, shuffle=False, num_workers=2)

# ---- models ----
class DetNet(nn.Module):  # pure deterministic: no dropout modules at all
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
    def forward(self, x): return self.model(x)

class MCNet(nn.Module):  # functional dropout via hooks (verified nonzero variance)
    def __init__(self, p=0.2):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1)
        self.p = p; self.armed = True
        for blk in self.model.decoder.blocks:
            blk.register_forward_hook(self._hook)
    def _hook(self, mod, inp, out): return F.dropout2d(out, p=self.p, training=self.armed)
    def forward(self, x): return self.model(x)

crit = CombinedBCEDiceLoss()

def run_train(net, seed, tag):
    seed_everything(seed)
    net.to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-6)
    scl = torch.cuda.amp.GradScaler()
    best, bp = -1, f"{tag}.pt"
    trace = []
    for ep in range(1, 11):
        net.train()
        if isinstance(net, MCNet): net.armed = True
        tl = 0; t0 = time.time()
        for b in tr_ld:
            x, y = b["image"].to(device), b["mask"].to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = crit(net(x), y)
            scl.scale(loss).backward(); scl.step(opt); scl.update(); tl += loss.item() * len(x)
        sch.step()
        net.eval()
        if isinstance(net, MCNet): net.armed = False
        vd_all, vd_pos = [], []
        with torch.no_grad():
            for b in va_ld:
                p = torch.sigmoid(net(b["image"].to(device))).cpu().numpy()
                y = b["mask"].numpy()
                for i in range(len(y)):
                    d = compute_dice_coefficient((y[i, 0] > 0).astype(np.uint8),
                                                 (p[i, 0] >= 0.5).astype(np.uint8))
                    vd_all.append(d)
                    if (y[i, 0] > 0).sum() > 0: vd_pos.append(d)
        m_all = float(np.mean(vd_all)); m_pos = float(np.mean(vd_pos)) if vd_pos else 0.0
        print(f"{tag} ep{ep}/10 [{time.time()-t0:.0f}s] loss={tl/len(tr_df):.4f} valDSC_all={m_all:.4f} valDSC_pos={m_pos:.4f}", flush=True)
        trace.append({"epoch": ep, "val_all": round(m_all, 4), "val_pos": round(m_pos, 4)})
        # VERBATIM original rule: best = argmax val DSC_pos (NOT all-case)
        if m_pos > best: best = m_pos; torch.save(net.state_dict(), bp)
    print(tag, "best:", best, flush=True)
    with open(f"{tag}_trace.json", "w") as f: json.dump(trace, f)
    return bp

p42 = run_train(DetNet(), 42, "det_seed42")
p43 = run_train(DetNet(), 43, "det_seed43")
p44 = run_train(DetNet(), 44, "det_seed44")
pmc = run_train(MCNet(), 42, "mc_seed42")

# ---- streaming eval ----
def load_net(cls, path, **kw):
    m = cls(**kw).to(device); m.load_state_dict(torch.load(path, map_location=device)); m.eval()
    return m
nets = {42: load_net(DetNet, p42), 43: load_net(DetNet, p43), 44: load_net(DetNet, p44)}
mc = load_net(MCNet, pmc); mc.armed = False
te_ld = DataLoader(SIIMPneumothoraxDataset(te_df, image_size=512, transforms=val_transforms),
                   batch_size=1, shuffle=False, num_workers=2)

def entropy(p): return -(p * np.log2(p + 1e-8) + (1 - p) * np.log2(1 - p + 1e-8))
def topk(a, k=500):
    f = a.ravel(); return float(f[np.argpartition(f, -k)[-k:]].mean())
N = len(te_df); S = 512 // 4
mm = np.memmap("/tmp/auc.dat", dtype=np.float32, mode="w+", shape=(3, N * S * S, 2))
B = 10
acc = {k: {"n": 0, "bn": np.zeros(B), "bp": np.zeros(B), "by": np.zeros(B), "br": 0.0} for k in ("det", "ens", "mc")}
def acc_up(a, p, y):
    b = np.clip((p * B).astype(int), 0, B - 1); a["n"] += y.size
    for i in range(B):
        m = b == i
        if m.any(): a["bn"][i] += m.sum(); a["bp"][i] += p[m].sum(); a["by"][i] += y[m].sum()
    a["br"] += float(((p - y) ** 2).sum())

fout = open("final_predictions.csv", "w")
fout.write("ImageId,PatientID,ViewPosition,HasPneumothorax,det_dice,mc_dice,ens_dice,det_unc,ens_unc,mc_unc,mc_var_mean\n")
t0 = time.time()
for j, b in enumerate(te_ld):
    x = b["image"].to(device); yn = b["mask"].numpy()[0, 0]
    with torch.no_grad():
        ps = [torch.sigmoid(nets[s](x))[0, 0].cpu().numpy() for s in (42, 43, 44)]
        em = np.stack(ps).mean(0)
        ee = entropy(em)
        me = np.stack([entropy(p) for p in ps]).mean(0)
        mi = np.clip(ee - me, 0, None)
        mc.armed = True
        mcs = np.stack([torch.sigmoid(mc(x))[0, 0].cpu().numpy() for _ in range(20)])
        mc.armed = False
        mmean = mcs.mean(0); mvar = mcs.var(0); ed = entropy(ps[0])
    dd = compute_segmentation_metrics(yn, ps[0]); de = compute_segmentation_metrics(yn, em)
    dm = compute_segmentation_metrics(yn, mmean)
    for k, pk, uk in (("det", ps[0], ed), ("ens", em, mi), ("mc", mmean, mvar)):
        acc_up(acc[k], pk, (yn > 0).astype(np.float32))
        err = np.abs((yn > 0).astype(np.float32) - (pk >= 0.5).astype(np.float32))
        us = uk[::4, ::4].ravel(); es = err[::4, ::4].ravel()
        mm[{"det": 0, "ens": 1, "mc": 2}[k], j * S * S:(j + 1) * S * S, 0] = us
        mm[{"det": 0, "ens": 1, "mc": 2}[k], j * S * S:(j + 1) * S * S, 1] = es
    r = te_df.iloc[j]
    fout.write(f"{r['ImageId']},{r['PatientID']},{r['ViewPosition']},{r['HasPneumothorax']},"
               f"{dd['dice']},{dm['dice']},{de['dice']},"
               f"{topk(ed)},{topk(mi)},{topk(mvar)},{float(mvar.mean())}\n")
    if (j + 1) % 400 == 0:
        fout.flush(); mm.flush(); print(f"{j+1}/{N} ({(time.time()-t0)/60:.1f}m)", flush=True)
fout.close(); mm.flush()

def esce_of(a):
    t = a["bn"].sum()
    return float((a["bn"] / t * np.abs(a["bp"] / np.maximum(a["bn"], 1) - a["by"] / np.maximum(a["bn"], 1))).sum())
OUT = {"esce": {k: round(esce_of(acc[k]), 6) for k in acc},
       "brier": {k: round(acc[k]["br"] / acc[k]["n"], 6) for k in acc}}
roc = {}
for i, k in enumerate(["det", "ens", "mc"]):
    U = np.array(mm[i, :, 0]); E = np.array(mm[i, :, 1]).astype(int)
    a = float(roc_auc_score(E, U)) if E.min() != E.max() else 0.5
    fpr, tpr, _ = roc_curve(E, U); kp = np.linspace(0, len(fpr) - 1, 200).astype(int)
    roc[k] = {"auc": round(a, 4), "fpr": [round(float(v), 4) for v in fpr[kp]],
              "tpr": [round(float(v), 4) for v in tpr[kp]]}
    print(k, "AUC:", round(a, 4), flush=True)
OUT["auroc_global"] = {k: roc[k]["auc"] for k in roc}
df = pd.read_csv("final_predictions.csv")
OUT["means"] = {c: round(float(df[c].mean()), 4) for c in ["det_dice", "mc_dice", "ens_dice", "mc_var_mean"]}
OUT["pos_means"] = {c: round(float(df[df["HasPneumothorax"] == 1][c].mean()), 4) for c in ["det_dice", "mc_dice", "ens_dice"]}
rng = np.random.default_rng(42); n = len(df)
for c in ["det_dice", "mc_dice", "ens_dice"]:
    v = df[c].values; bs = v[rng.integers(0, n, (1000, n))].mean(1)
    OUT[c + "_ci95"] = [round(float(np.percentile(bs, 2.5)), 4), round(float(np.percentile(bs, 97.5)), 4)]
OUT["wilcoxon"] = {"mc_det_p": float(wilcoxon(df["mc_dice"], df["det_dice"], alternative="two-sided").pvalue),
                   "ens_det_p": float(wilcoxon(df["ens_dice"], df["det_dice"], alternative="two-sided").pvalue)}
# standard AURC from CSV
for m, u in (("det", "det_unc"), ("ens", "ens_unc"), ("mc", "mc_unc")):
    o = np.argsort(df[u].values, kind="stable"); d = df[m + "_dice"].values[o]
    cum = np.cumsum(d) / np.arange(1, n + 1); cov = np.arange(1, n + 1) / n
    OUT[m + "_aurc_std"] = round(float(np.trapezoid(1 - cum, cov)), 4)
with open("final_metrics.json", "w") as f: json.dump(OUT, f, indent=2)
print(json.dumps(OUT, indent=2))
fig, ax = plt.subplots(figsize=(6.5, 5), dpi=300)
for m, u, col in (("det", "det_unc", "#e74c3c"), ("ens", "ens_unc", "#2980b9"), ("mc", "mc_unc", "#f39c12")):
    o = np.argsort(df[u].values, kind="stable"); d = df[m + "_dice"].values[o]
    cum = np.cumsum(d) / np.arange(1, n + 1); cov = np.arange(1, n + 1) / n
    ax.plot(cov, 1 - cum, color=col, linewidth=2.0, label=f"{m} ({OUT[m+'_aurc_std']})")
ax.set_xlabel("Coverage", fontweight="bold"); ax.set_ylabel("Risk (1-DSC_all)", fontweight="bold")
ax.set_title("Risk-Coverage (standard, final run)", fontweight="bold"); ax.legend()
plt.tight_layout(); plt.savefig("fig1_rc.png", dpi=300); plt.close()
fig, ax = plt.subplots(figsize=(6.5, 5), dpi=300)
cols = {"det": "#e74c3c", "ens": "#2980b9", "mc": "#f39c12"}
ax.plot([0, 1], [0, 1], "k--", label="Chance")
for k in ("det", "ens", "mc"):
    ax.plot(roc[k]["fpr"], roc[k]["tpr"], color=cols[k], linewidth=2.0, label=f"{k} ({roc[k]['auc']})")
ax.set_xlabel("FPR", fontweight="bold"); ax.set_ylabel("TPR", fontweight="bold")
ax.set_title("Empirical ROC (global pixels, final run)", fontweight="bold")
ax.legend(loc="lower right"); plt.tight_layout(); plt.savefig("fig3_roc.png", dpi=300)
print("saved all")
